<center><img src="trainers.jpg" alt="Trainers in a store" width=550></center>

Sports clothing and athleisure attire is a huge industry, worth approximately $193 billion in 2021 with a strong growth forecast over the next decade! (`Source: https://www.statista.com/statistics/254489/total-revenue-of-the-global-sports-apparel-market/`)

In this workbook, you will undertake the role of a product analyst for an online sports clothing company. The company is specifically interested in how it can improve revenue. You will dive into product data such as pricing, reviews, descriptions, and ratings, as well as revenue and website traffic, to produce recommendations for its marketing and sales teams.  

### The data:
You've been provided with four datasets to investigate:

`brands.csv`

| Columns | Description |
|---------|-------------|
| `product_id` | Unique product identifier |
| `brand` | Brand of the product | 

`finance.csv`

| Columns | Description |
|---------|-------------|
| `product_id` | Unique product identifier |
| `listing_price` | Original price of the product | 
| `sale_price` | Discounted price of the product |
| `discount` | Discount off the listing price, as a decimal | 
| `revenue` | Revenue generated by the product |

`info.csv`

| Columns | Description |
|---------|-------------|
| `product_name` | Name of the product | 
| `product_id` | Unique product identifier |
| `description` | Description of the product |

`reviews.csv`

| Columns | Description |
|---------|-------------|
| `product_id` | Unique product identifier |
| `rating` | Average product rating | 
| `reviews` | Number of reviews for the product |

In [69]:
# Importing libraries
import pandas as pd

# Loading the data
brands = pd.read_csv("brands.csv") 
finance = pd.read_csv("finance.csv")
info = pd.read_csv("info.csv")
reviews = pd.read_csv("reviews.csv")

# Start coding here...

In [70]:
def merge_dataset(dataset_list) :
    left_dataset = dataset_list[0]
    for dataset_index in range(0,len(dataset_list) - 1) :
        sport_data = pd.merge(left=left_dataset,
                              right=dataset_list[dataset_index + 1],
                              how="inner",
                              on=["product_id"]
        )
        left_dataset = sport_data
      
    return sport_data
         
merged_data = merge_dataset([brands, finance, info, reviews])

print(merged_data.head(10))

  product_id   brand  ...  rating  reviews
0     AH2430     NaN  ...     NaN      NaN
1     G27341  Adidas  ...     3.3     24.0
2     CM0081  Adidas  ...     2.6     37.0
3     B44832  Adidas  ...     4.1     35.0
4     D98205  Adidas  ...     3.5     72.0
5     B75586  Adidas  ...     1.0     45.0
6     CG4051  Adidas  ...     4.4      2.0
7     CM0080  Adidas  ...     2.8      7.0
8     B75990  Adidas  ...     4.5     16.0
9     EE5761  Adidas  ...     4.0     39.0

[10 rows x 10 columns]


In [71]:
print(merged_data.isna().sum())

threshold = len(merged_data) * 0.05
print(threshold)

cols_to_drop = merged_data.columns[merged_data.isna().sum() <= threshold]
print(cols_to_drop)

merged_data.dropna(subset=cols_to_drop, inplace=True)

product_id        0
brand            59
listing_price    59
sale_price       59
discount         59
revenue          59
product_name     59
description      62
rating           59
reviews          59
dtype: int64
158.95000000000002
Index(['product_id', 'brand', 'listing_price', 'sale_price', 'discount',
       'revenue', 'product_name', 'description', 'rating', 'reviews'],
      dtype='object')


In [72]:
price_labels = ["Budget", "Average", "Expensive", "Elite"]

Q1 = merged_data["listing_price"].quantile(0.25)
Q2 = merged_data["listing_price"].median()
Q3 = merged_data["listing_price"].quantile(0.75)  # Fixed missing parenthesis

price_bins = [merged_data["listing_price"].min()-1, Q1, Q2, Q3, merged_data["listing_price"].max()+1]
merged_data["price_label"] = pd.cut(merged_data["listing_price"], bins=price_bins, labels=price_labels)

print(merged_data[["sale_price", "price_label"]])  # Fixed column selection syntax

      sale_price price_label
1          37.99   Expensive
2           5.99      Budget
3          34.99   Expensive
4          39.99   Expensive
5          19.20     Average
...          ...         ...
3174       64.95      Budget
3175      139.95      Budget
3176      127.97       Elite
3177      169.95      Budget
3178       62.97   Expensive

[3117 rows x 2 columns]


In [73]:
print(merged_data["brand"].value_counts())

Adidas    2575
Nike       542
Name: brand, dtype: int64


In [74]:
adidas_vs_nike = merged_data.groupby(["brand", "price_label"]).agg(
    num_products = ("product_id", "count"),
    mean_revenue = ("revenue", "mean")
).reset_index()

adidas_vs_nike = adidas_vs_nike.round(2)

print(adidas_vs_nike)                                                                       

    brand price_label  num_products  mean_revenue
0  Adidas      Budget           574       2015.68
1  Adidas     Average           655       3035.30
2  Adidas   Expensive           759       4621.56
3  Adidas       Elite           587       8302.78
4    Nike      Budget           357       1596.33
5    Nike     Average             8        675.59
6    Nike   Expensive            47        500.56
7    Nike       Elite           130       1367.45


In [75]:
print(merged_data["description"])

1       A modern take on adidas sport heritage, tailor...
2       These adidas Puka slippers for women's come wi...
3       Inspired by modern tech runners, these women's...
4       This design is inspired by vintage Taekwondo s...
5       Refine your interval training in these women's...
                              ...                        
3174    The Nike Tiempo Legend 8 Academy TF takes the ...
3175    The Nike React Metcon AMP takes the stability ...
3176    The Air Jordan 8 Retro recaptures the memorabl...
3177    The Nike Air Max 98 features the OG design lin...
3178    A mash-up of Pegasus' past, the Nike P-6000 SE...
Name: description, Length: 3117, dtype: object


In [76]:
merged_data["description_len"] = merged_data["description"].str.len()

# Créer des intervalles de 100 caractères
bins = range(0, merged_data["description_len"].max() + 100, 100)
labels = [f"{i+100}" for i in bins[:-1]]
merged_data["description_length"] = pd.cut(merged_data["description_len"], bins=bins, labels=labels)

print(merged_data[["description_len","description_length"]])

      description_len description_length
1                 175                200
2                 172                200
3                 264                300
4                 288                300
5                 221                300
...               ...                ...
3174              146                200
3175              378                400
3176              204                300
3177              240                300
3178              202                300

[3117 rows x 2 columns]


In [77]:
description_lengths = merged_data.groupby("description_length").agg(
    mean_rating = ("rating", "mean"),
    total_reviews = ("reviews", "sum")
).reset_index()

description_lengths = description_lengths.round(2)

print(description_lengths)

  description_length  mean_rating  total_reviews
0                100         2.26           36.0
1                200         3.19        17719.0
2                300         3.28        76115.0
3                400         3.29        28994.0
4                500         3.35         4984.0
5                600         3.12          852.0
6                700         3.65          818.0
